# Task 9: Summary Statistics & Automated NLP Insights

**Objective:** Develop a custom report generator that:
- Computes comprehensive statistics (mean, median, variance, skewness, kurtosis, percentiles, CV)
- Interprets trends using a rule-based NLP engine to produce **human-readable summaries**
- Generates an automated HTML report with embedded narrative insights

**Input:** `data/processed/womens_clothing_scaled.csv`

## 1. Setup & Imports

In [ ]:
import os
import json
import logging
import warnings
import textwrap
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')

# ── Logging setup ──────────────────────────────────────────────────────────────
os.makedirs('../../logs', exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('../../logs/task_9.log', mode='w'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('Task9')
logger.info('Task 9 started.')

# ── Style ───────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  '#e0e0ff',
    'xtick.color':      '#c0c0e0',
    'ytick.color':      '#c0c0e0',
    'text.color':       '#e0e0ff',
    'grid.color':       '#2a2a4a',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'font.size':        10,
    'axes.titlesize':   12,
    'axes.titleweight': 'bold',
    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
})

PALETTE = ['#7c5cbf', '#e96479', '#f5a623', '#4fc3f7', '#81c784', '#ff8a65']
os.makedirs('../../reports', exist_ok=True)

print('✅ Setup complete.')

## 2. Load Data

In [ ]:
df = pd.read_csv('../../data/processed/womens_clothing_scaled.csv')
logger.info(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Shape: {df.shape}')
df.head()

## 3. Custom Statistics Engine

In [ ]:
def compute_stats(series: pd.Series) -> dict:
    """Compute a comprehensive statistics dictionary for a pandas Series."""
    s = series.dropna()
    if len(s) == 0:
        return {'count': 0, 'missing': len(series), 'dtype': str(series.dtype)}

    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outlier_count = int(((s < lower_fence) | (s > upper_fence)).sum())

    result = {
        'dtype':         str(series.dtype),
        'count':         int(len(s)),
        'missing':       int(len(series) - len(s)),
        'missing_pct':   round((len(series) - len(s)) / len(series) * 100, 2),
        'unique':        int(s.nunique()),
        'mean':          round(float(s.mean()), 4),
        'median':        round(float(s.median()), 4),
        'mode':          round(float(s.mode().iloc[0]), 4) if s.dtype.kind in 'biufc' else str(s.mode().iloc[0]),
        'std':           round(float(s.std()), 4),
        'variance':      round(float(s.var()), 4),
        'min':           round(float(s.min()), 4),
        'max':           round(float(s.max()), 4),
        'range':         round(float(s.max() - s.min()), 4),
        'q1':            round(float(q1), 4),
        'q3':            round(float(q3), 4),
        'iqr':           round(float(iqr), 4),
        'p5':            round(float(s.quantile(0.05)), 4),
        'p95':           round(float(s.quantile(0.95)), 4),
        'cv_pct':        round(float(s.std() / s.mean() * 100), 2) if s.mean() != 0 else None,
        'skewness':      round(float(s.skew()), 4),
        'kurtosis':      round(float(s.kurtosis()), 4),
        'outlier_count': outlier_count,
        'outlier_pct':   round(outlier_count / len(s) * 100, 2),
        'lower_fence':   round(float(lower_fence), 4),
        'upper_fence':   round(float(upper_fence), 4),
    }
    return result

print('✅ Statistics engine ready.')

## 4. NLP Interpretation Engine

In [ ]:
def interpret_skewness(skew: float) -> str:
    """Natural language interpretation of skewness."""
    if abs(skew) < 0.2:
        return 'The distribution is approximately symmetric with minimal skew.'
    elif 0.2 <= skew < 0.5:
        return 'The distribution shows a mild positive (right) skew, with a slight pull toward higher values.'
    elif 0.5 <= skew < 1.0:
        return 'The distribution is moderately positively skewed, indicating a longer tail on the right side.'
    elif skew >= 1.0:
        return f'The distribution is highly positively skewed (skew={skew:.2f}), suggesting many low values with rare extreme high values.'
    elif -0.5 < skew <= -0.2:
        return 'The distribution shows a mild negative (left) skew, with a slight pull toward lower values.'
    elif -1.0 < skew <= -0.5:
        return 'The distribution is moderately negatively skewed, with a longer tail on the left side.'
    else:
        return f'The distribution is highly negatively skewed (skew={skew:.2f}), suggesting many high values with rare extreme low values.'

def interpret_kurtosis(kurt: float) -> str:
    """Natural language interpretation of excess kurtosis."""
    if abs(kurt) < 0.5:
        return 'The distribution has near-normal tailedness (mesokurtic), consistent with a Gaussian-like distribution.'
    elif kurt >= 3:
        return f'The distribution is highly leptokurtic (kurtosis={kurt:.2f}), indicating very heavy tails and a sharp central peak — extreme values occur more frequently than expected.'
    elif 0.5 <= kurt < 3:
        return f'The distribution is leptokurtic (kurtosis={kurt:.2f}), with heavier tails than normal, suggesting occasional extreme values.'
    elif -0.5 >= kurt > -3:
        return f'The distribution is platykurtic (kurtosis={kurt:.2f}), with lighter tails than normal — values are more uniformly spread.'
    else:
        return f'The distribution is highly platykurtic (kurtosis={kurt:.2f}), indicating a very flat distribution with minimal extreme values.'

def interpret_outliers(stat: dict) -> str:
    """Narrate outlier presence."""
    pct = stat.get('outlier_pct', 0)
    n = stat.get('outlier_count', 0)
    if pct == 0:
        return 'No IQR-based outliers detected — the data appears well-behaved within expected bounds.'
    elif pct < 2:
        return f'A small number of outliers ({n} values, {pct:.1f}%) were detected using the IQR fence method, which is typical in real-world data.'
    elif pct < 5:
        return f'{n} outliers ({pct:.1f}%) were found — a moderate level suggesting some data quality concerns or genuine extreme observations.'
    else:
        return f'A high proportion of outliers ({n} values, {pct:.1f}%) detected — this feature may contain noisy data or multi-modal distributions requiring special treatment.'

def interpret_cv(cv: float | None) -> str:
    """Narrate coefficient of variation."""
    if cv is None:
        return 'Coefficient of variation is undefined (mean is zero).'
    if cv < 15:
        return f'Low variability (CV={cv:.1f}%): values are tightly clustered around the mean.'
    elif cv < 30:
        return f'Moderate variability (CV={cv:.1f}%): a reasonable spread relative to the mean.'
    else:
        return f'High variability (CV={cv:.1f}%): large spread relative to the mean — this feature is quite dispersed.'

def generate_column_narrative(col_name: str, stat: dict) -> str:
    """Generate a full natural language narrative for a single column."""
    lines = [
        f"### {col_name}",
        f"This column contains **{stat['count']:,} non-null values** out of {stat['count'] + stat['missing']:,} total "
        f"({'no missing values' if stat['missing'] == 0 else f"{stat['missing']:,} missing ({stat['missing_pct']:.1f}%)"}). "
        f"It has **{stat['unique']:,} unique values**.",
        "",
        f"**Central Tendency:** The mean is **{stat['mean']}**, the median is **{stat['median']}**, "
        f"and the mode is **{stat['mode']}**. "
        + ("Mean and median are close, suggesting limited influence from extreme values."
           if abs(stat['mean'] - stat['median']) < 0.1 * max(abs(stat['mean']), 1)
           else "A notable gap between mean and median indicates the presence of skewing outliers."),
        "",
        f"**Dispersion:** Standard deviation = **{stat['std']}**, IQR = **{stat['iqr']}** "
        f"(Q1={stat['q1']}, Q3={stat['q3']}). Range: [{stat['min']} – {stat['max']}].",
        interpret_cv(stat.get('cv_pct')),
        "",
        f"**Distribution Shape:** {interpret_skewness(stat['skewness'])}",
        interpret_kurtosis(stat['kurtosis']),
        "",
        f"**Outliers:** {interpret_outliers(stat)}",
    ]
    return '\n'.join(lines)

print('✅ NLP interpretation engine ready.')

## 5. Compute Full Statistics for All Columns

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Processing {len(num_cols)} numerical columns...')

all_stats = {}
for col in num_cols:
    all_stats[col] = compute_stats(df[col])
    logger.info(f'  Stats computed for: {col}')

# ── Build summary DataFrame ────────────────────────────────────────────────────
stats_df = pd.DataFrame(all_stats).T.reset_index().rename(columns={'index': 'Column'})

# Convert numeric columns
numeric_stat_cols = ['count','missing','missing_pct','unique','mean','median','std','variance',
                     'min','max','range','q1','q3','iqr','p5','p95','cv_pct',
                     'skewness','kurtosis','outlier_count','outlier_pct','lower_fence','upper_fence']
for c in numeric_stat_cols:
    if c in stats_df.columns:
        stats_df[c] = pd.to_numeric(stats_df[c], errors='coerce')

stats_df.to_csv('../../reports/task_9_summary_statistics.csv', index=False)
logger.info('Summary statistics CSV saved.')
print(f'✅ Statistics computed for {len(num_cols)} columns.')
display(stats_df.head(10))

## 6. Summary Dashboard Visualization

In [ ]:
# ── Box-Violin hybrid plots ────────────────────────────────────────────────────
plot_cols = num_cols[:min(8, len(num_cols))]
n_cols_plot = min(4, len(plot_cols))
n_rows_plot = (len(plot_cols) + n_cols_plot - 1) // n_cols_plot

fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                         figsize=(n_cols_plot * 4, n_rows_plot * 4))
fig.suptitle('Distribution Summary – Box + Violin Hybrid', fontsize=14, color='#e0e0ff', y=1.01)
axes = np.array(axes).ravel()

for i, col in enumerate(plot_cols):
    ax = axes[i]
    data = df[col].dropna()
    # Violin
    parts = ax.violinplot(data, positions=[0], showmedians=False, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(PALETTE[i % len(PALETTE)])
        pc.set_alpha(0.5)
        pc.set_edgecolor('white')
    # Box
    bp = ax.boxplot(data, positions=[0], widths=0.15, patch_artist=True,
                    medianprops={'color': '#f5a623', 'linewidth': 2},
                    boxprops={'facecolor': PALETTE[i % len(PALETTE)], 'alpha': 0.8},
                    whiskerprops={'color': '#e0e0ff'},
                    capprops={'color': '#e0e0ff'},
                    flierprops={'marker': 'o', 'color': '#e96479',
                                'markersize': 3, 'alpha': 0.5})
    ax.set_title(col, fontsize=10)
    ax.set_xticks([])
    ax.yaxis.grid(True, alpha=0.3)
    # Annotate stats
    s = all_stats[col]
    ax.text(0.98, 0.98, f"μ={s['mean']}\nσ={s['std']}\nskew={s['skewness']}",
            transform=ax.transAxes, ha='right', va='top',
            fontsize=7.5, color='#f5a623',
            bbox=dict(facecolor='#1a1a2e', alpha=0.7, edgecolor='none', pad=2))

for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig('../../reports/task_9_box_violin_summary.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('Box-violin summary plot saved.')

In [ ]:
# ── Skewness & Kurtosis comparison bar charts ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Shape Statistics Comparison Across Features', fontsize=13, color='#e0e0ff')

skew_vals = stats_df['skewness'].values
kurt_vals = stats_df['kurtosis'].values
col_names = stats_df['Column'].values

skew_colors = ['#e96479' if abs(v) > 1 else '#7c5cbf' if abs(v) > 0.5 else '#4fc3f7'
               for v in skew_vals]
axes[0].barh(col_names, skew_vals, color=skew_colors, edgecolor='white', linewidth=0.4)
axes[0].axvline(0, color='white', linewidth=1.2)
axes[0].axvline( 0.5, color='#f5a623', linewidth=0.8, linestyle='--', alpha=0.7)
axes[0].axvline(-0.5, color='#f5a623', linewidth=0.8, linestyle='--', alpha=0.7)
axes[0].set_title('Skewness per Feature'); axes[0].set_xlabel('Skewness')
axes[0].xaxis.grid(True, alpha=0.3)

kurt_colors = ['#e96479' if v > 3 else '#81c784' if v < -1 else '#7c5cbf'
               for v in kurt_vals]
axes[1].barh(col_names, kurt_vals, color=kurt_colors, edgecolor='white', linewidth=0.4)
axes[1].axvline(0, color='white', linewidth=1.2)
axes[1].set_title('Excess Kurtosis per Feature'); axes[1].set_xlabel('Kurtosis')
axes[1].xaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../reports/task_9_skewness_kurtosis.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('Skewness & kurtosis chart saved.')

In [ ]:
# ── Summary Statistics Heatmap ─────────────────────────────────────────────────
heat_metrics = ['mean', 'median', 'std', 'skewness', 'kurtosis', 'outlier_pct', 'cv_pct']
heat_data = stats_df[['Column'] + heat_metrics].set_index('Column').astype(float)

# Normalize each column to [0,1] for color scaling
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min() + 1e-9)

fig, ax = plt.subplots(figsize=(max(10, len(heat_metrics) * 1.5), max(6, len(stats_df) * 0.5)))
fig.patch.set_facecolor('#0f0f1a')

sns.heatmap(heat_norm, annot=heat_data.round(2), fmt='.2f',
            cmap='plasma', ax=ax,
            linewidths=0.5, linecolor='#0f0f1a',
            annot_kws={'size': 9},
            cbar_kws={'label': 'Normalized Value'})
ax.set_title('Summary Statistics Heatmap – All Features', fontsize=13)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.savefig('../../reports/task_9_stats_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('Summary stats heatmap saved.')

## 7. Generate Insights JSON

In [ ]:
insights = {}
for col in num_cols:
    s = all_stats[col]
    insights[col] = {
        'statistics': s,
        'narrative': {
            'skewness_insight':  interpret_skewness(s['skewness']),
            'kurtosis_insight':  interpret_kurtosis(s['kurtosis']),
            'outlier_insight':   interpret_outliers(s),
            'cv_insight':        interpret_cv(s.get('cv_pct')),
        }
    }

insights_path = '../../reports/task_9_insights_report.json'
with open(insights_path, 'w') as f:
    json.dump(insights, f, indent=2, default=str)

logger.info('Insights JSON report saved.')
print(f'✅ Insights JSON saved with {len(insights)} column entries.')

## 8. Generate Automated HTML Report

In [ ]:
def build_html_report(all_stats: dict, insights: dict, df_shape: tuple) -> str:
    """Build a full standalone HTML report with narrative insights."""
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    n_rows, n_cols_total = df_shape

    # ── Column cards ──────────────────────────────────────────────────────────
    cards_html = ''
    for col, s in all_stats.items():
        narr = insights.get(col, {}).get('narrative', {})
        skew_color = ('#e96479' if abs(s.get('skewness', 0)) > 1
                      else '#f5a623' if abs(s.get('skewness', 0)) > 0.5
                      else '#81c784')
        cards_html += f"""
        <div class="card">
          <div class="card-header">
            <span class="col-name">{col}</span>
            <span class="badge" style="background:{skew_color}">
              skew={s.get('skewness','N/A')}
            </span>
          </div>
          <div class="stat-grid">
            <div class="stat-item"><span class="stat-label">Count</span><span class="stat-value">{s.get('count','N/A'):,}</span></div>
            <div class="stat-item"><span class="stat-label">Missing</span><span class="stat-value">{s.get('missing_pct','N/A')}%</span></div>
            <div class="stat-item"><span class="stat-label">Mean</span><span class="stat-value">{s.get('mean','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Median</span><span class="stat-value">{s.get('median','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Std Dev</span><span class="stat-value">{s.get('std','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Variance</span><span class="stat-value">{s.get('variance','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Min</span><span class="stat-value">{s.get('min','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Max</span><span class="stat-value">{s.get('max','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Q1</span><span class="stat-value">{s.get('q1','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Q3</span><span class="stat-value">{s.get('q3','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">IQR</span><span class="stat-value">{s.get('iqr','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">CV%</span><span class="stat-value">{s.get('cv_pct','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Skewness</span><span class="stat-value">{s.get('skewness','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Kurtosis</span><span class="stat-value">{s.get('kurtosis','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">P5</span><span class="stat-value">{s.get('p5','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">P95</span><span class="stat-value">{s.get('p95','N/A')}</span></div>
            <div class="stat-item"><span class="stat-label">Outliers</span><span class="stat-value">{s.get('outlier_count','N/A')} ({s.get('outlier_pct','N/A')}%)</span></div>
            <div class="stat-item"><span class="stat-label">Unique</span><span class="stat-value">{s.get('unique','N/A')}</span></div>
          </div>
          <div class="narrative">
            <div class="narr-item">📐 <em>{narr.get('skewness_insight','')}</em></div>
            <div class="narr-item">📊 <em>{narr.get('kurtosis_insight','')}</em></div>
            <div class="narr-item">⚠️ <em>{narr.get('outlier_insight','')}</em></div>
            <div class="narr-item">📉 <em>{narr.get('cv_insight','')}</em></div>
          </div>
        </div>
        """

    html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Task 9 – Automated Summary Statistics Report</title>
  <style>
    * {{ box-sizing: border-box; margin: 0; padding: 0; }}
    body {{
      background: #0f0f1a;
      color: #e0e0ff;
      font-family: 'Segoe UI', system-ui, -apple-system, sans-serif;
      font-size: 14px;
      line-height: 1.6;
      padding: 32px;
    }}
    h1 {{ font-size: 2rem; margin-bottom: 4px;
          background: linear-gradient(90deg, #7c5cbf, #e96479, #f5a623);
          -webkit-background-clip: text; -webkit-text-fill-color: transparent; }}
    .subtitle {{ color: #9090b0; margin-bottom: 24px; }}
    .meta {{ display: flex; gap: 24px; margin-bottom: 32px; flex-wrap: wrap; }}
    .meta-item {{
      background: #1a1a2e;
      border: 1px solid #333355;
      border-radius: 10px;
      padding: 12px 20px;
      text-align: center;
    }}
    .meta-item .val {{ font-size: 1.5rem; font-weight: 700; color: #7c5cbf; }}
    .meta-item .lbl {{ font-size: 0.75rem; color: #9090b0; }}
    .cards {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(480px, 1fr)); gap: 20px; }}
    .card {{
      background: #1a1a2e;
      border: 1px solid #2a2a4a;
      border-radius: 14px;
      padding: 20px;
      transition: box-shadow 0.2s;
    }}
    .card:hover {{ box-shadow: 0 0 24px rgba(124,92,191,0.3); }}
    .card-header {{ display: flex; justify-content: space-between; align-items: center;
                    margin-bottom: 14px; }}
    .col-name {{ font-size: 1.05rem; font-weight: 700; color: #c0a0ff; }}
    .badge {{ font-size: 0.72rem; padding: 3px 10px; border-radius: 20px;
              color: white; font-weight: 600; }}
    .stat-grid {{
      display: grid;
      grid-template-columns: repeat(3, 1fr);
      gap: 8px;
      margin-bottom: 14px;
    }}
    .stat-item {{
      background: #12122a;
      border-radius: 8px;
      padding: 8px 10px;
      display: flex; flex-direction: column;
    }}
    .stat-label {{ font-size: 0.68rem; color: #7070a0; text-transform: uppercase;
                   letter-spacing: 0.5px; }}
    .stat-value {{ font-size: 0.95rem; font-weight: 600; color: #e0e0ff; }}
    .narrative {{
      background: #12122a;
      border-left: 3px solid #7c5cbf;
      border-radius: 0 8px 8px 0;
      padding: 12px 14px;
      display: flex; flex-direction: column; gap: 6px;
    }}
    .narr-item {{ font-size: 0.82rem; color: #c0c0e0; }}
    footer {{ text-align: center; margin-top: 40px; color: #5050a0; font-size: 0.8rem; }}
  </style>
</head>
<body>
  <h1>Automated Summary Statistics &amp; Insights</h1>
  <div class="subtitle">Women's Clothing E-Commerce ML Project — Task 9 Auto-Report</div>
  <div class="meta">
    <div class="meta-item"><div class="val">{n_rows:,}</div><div class="lbl">Total Rows</div></div>
    <div class="meta-item"><div class="val">{n_cols_total}</div><div class="lbl">Total Columns</div></div>
    <div class="meta-item"><div class="val">{len(all_stats)}</div><div class="lbl">Numerical Features Analyzed</div></div>
    <div class="meta-item"><div class="val">{now}</div><div class="lbl">Generated At</div></div>
  </div>
  <div class="cards">
    {cards_html}
  </div>
  <footer>Auto-generated by Task 9 NLP Report Engine · Women's Clothing ML Project</footer>
</body>
</html>
    """
    return html

html_report = build_html_report(all_stats, insights, df.shape)
html_path = '../../reports/task_9_automated_report.html'
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(html_report)

logger.info('Automated HTML report saved.')
print(f'✅ HTML report saved to: {html_path}')

## 9. Print Sample Narratives

In [ ]:
# Print narrative for first 3 columns as sample output
for col in num_cols[:3]:
    print('=' * 70)
    print(generate_column_narrative(col, all_stats[col]))
    print()

logger.info('Task 9 complete.')
print('=' * 70)
print('✅ Task 9 complete. All reports saved to reports/ folder.')
print()
print('Output files:')
print('  📄 reports/task_9_summary_statistics.csv')
print('  📄 reports/task_9_insights_report.json')
print('  🌐 reports/task_9_automated_report.html')
print('  🖼️  reports/task_9_box_violin_summary.png')
print('  🖼️  reports/task_9_skewness_kurtosis.png')
print('  🖼️  reports/task_9_stats_heatmap.png')